In [2]:
from pathlib import Path
import pyvista as pv
import numpy as np


from phd_helpers.MeshQuality import check_mesh_quality, mesh_quality_summary, plot_bad_cells2
from phd_helpers.paths import get_mesh, get_subject_stl_path

In [3]:
def extract_3Dmesh_regions(mesh):
    """mesh with bone(1) cartilage(2) volumes and bone(-1) cartilage(-2) interface(-3) surfaces"""
    shell_mesh = mesh.extract_cells_by_type(5)
    return {
        'mesh': mesh,

        'shell': shell_mesh.extract_surface(algorithm=None),
        'bone_shell': shell_mesh.extract_cells(np.where(shell_mesh['region_id']==-2)[0], invert=True).extract_surface(algorithm=None),
        'cartilage_shell': shell_mesh.extract_cells(np.where(shell_mesh['region_id']==-1)[0], invert=True).extract_surface(algorithm=None),

        'bone_surf': shell_mesh.extract_cells(np.where(shell_mesh['region_id']==-1)[0]).extract_surface(algorithm=None),
        'cartilage_surf': shell_mesh.extract_cells(np.where(shell_mesh['region_id']==-2)[0]).extract_surface(algorithm=None),
        'interface_surf': shell_mesh.extract_cells(np.where(shell_mesh['region_id']==-3)[0]).extract_surface(algorithm=None),

        'bone_vol': mesh.extract_cells(np.where(mesh['region_id']==1)[0]),
        'cartilage_vol': mesh.extract_cells(np.where(mesh['region_id']==2)[0])
    }

In [ ]:
subject, sideL = '14548', 'R'
stl_path = get_subject_stl_path(subject, sideL)

bone_arbone = 'tpm-mc1'
bone = bone_arbone.split('-')[0]
id_1, id_2 = 0, 0
id_2d = f'-{id_1}'
id_cart = f'-{id_1}-{id_2}'

id_3 = 0
id_3d = f'-{id_1}-{id_2}-{id_3}'
study_id = 'a'

root_dir = Path('../../../../MeshPipeline/outputs/ParamOptimisation/criteria3D')

path_mesh = root_dir / f'study1{study_id}/meshes/{subject}{sideL}/{bone_arbone}'
path2d = path_mesh / '2Dmesh'
path3d = path_mesh / '3Dmesh'

# combined
mesh3d = pv.read(path3d / f'mesh{id_3d}.vtu')
mesh_dict = extract_3Dmesh_regions(mesh3d)

# cartilage
cart_orig = pv.read(path2d / f'orig_cart_surf{id_cart}.vtp') # inner cells
cart_smooth = pv.read(path2d / f'smooth_cart_surf{id_cart}.vtp') # inner cells
# cart remesh
mesh2d = pv.read(path2d / f'bone_cartilage_mesh{id_cart}.vtp')
cart_remesh2d = mesh2d.extract_cells(np.where(mesh2d['region_id']==2)[0]) # inner cells
cart_remesh3d = mesh_dict['cartilage_surf'] # no inner cells

# bone
bone_orig = get_mesh(stl_path, bone) # no inner cells
bone_smooth = pv.read(path2d / f'bone_smooth{id_2d}.obj') # no inner cells
# bone_remesh2d = pv.read(path2d / f'bone_remesh{id_2d}.obj') # no inner cells
bone_remesh2d = mesh2d.extract_cells(np.where(mesh2d['region_id']==2)[0], invert=True).extract_surface(algorithm=None) # region id
bone_remesh3d = mesh_dict['bone_shell'] # region id

# should also get the bone interface region on all bone meshes
# interface


# always use ['inner_points'] for distance meaures
